# UDCF (LBCF, Ai et al. 2022, WWW'22) aplicado a base Hillstrom

Este notebook usa o **codigo C++ original dos autores** (repositorio publico do artigo), sem reimplementacao, para estimar o CATE (Conditional Average Treatment Effect) na base publica Hillstrom.

Etapa coberta: apenas a estimacao de CATE via UDCF (Secao 4.2.1 do artigo). A segunda etapa do LBCF (otimizacao com orcamento via DGB) nao esta incluida.

Basta rodar as celulas em ordem (Ambiente de execucao > Executar tudo).

## 1. Clonar o repositorio dos autores e instalar ferramentas de build

In [ ]:
!git clone -q https://github.com/www2022paper/WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS.git
!apt-get -qq update && apt-get -qq install -y cmake g++

## 2. Extrair o codigo C++ do UDCF (vem zipado dentro do repositorio)

In [ ]:
import zipfile

BASE = "WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/LBCF"
with zipfile.ZipFile(f"{BASE}/LBCF_RCT.zip") as z:
    z.extractall(BASE)

## 3. Carregar a base Hillstrom e montar o arquivo de entrada

Escolhas de dados (validadas previamente):
- Outcome (Y): `conversion`
- Tratamento multi-nivel (K=2): `No E-Mail` = controle, `Mens E-Mail` = T1, `Womens E-Mail` = T2
- Features: `recency, history, mens, womens, newbie` + dummies de `zip_code` e `channel`

O C++ dos autores espera um arquivo de texto sem cabecalho, valores separados por espaco, com as colunas na ordem `[features..., outcome, tratamento_1, tratamento_2]`.

In [ ]:
import pandas as pd

HILLSTROM_URL = (
    "http://www.minethatdata.com/"
    "Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
)
df = pd.read_csv(HILLSTROM_URL)
print("Base Hillstrom original:", df.shape)

feature_cols = ["recency", "history", "mens", "womens", "newbie"]
X = df[feature_cols].astype(float).copy()
zip_dummies = pd.get_dummies(df["zip_code"], prefix="zip", dtype=float)
channel_dummies = pd.get_dummies(df["channel"], prefix="channel", dtype=float)
X = pd.concat([X, zip_dummies, channel_dummies], axis=1)

Y = df["conversion"].astype(float)
code = df["segment"].map({"No E-Mail": 0, "Mens E-Mail": 1, "Womens E-Mail": 2})
T1 = (code == 1).astype(float)
T2 = (code == 2).astype(float)

design = pd.concat([X, Y.rename("Y"), T1.rename("T1"), T2.rename("T2")], axis=1)

n_features = X.shape[1]
outcome_index = n_features
treatment_index = [n_features + 1, n_features + 2]

data_path = f"{BASE}/UDCF_RCT/core/hillstrom_udcf_input.txt"
design.to_csv(data_path, sep=" ", header=False, index=False)

print("outcome_index:", outcome_index, "| treatment_index:", treatment_index)
print("linhas x colunas do arquivo de entrada:", design.shape)

## 4. Gerar o `main.cpp` adaptado

So os indices de coluna e os caminhos de arquivo mudam. O treinamento, a divisao (split) do UDCF e a predicao sao exatamente o codigo original dos autores (`udcf_trainer`, `udcf_predictor`, hiperparametros padrao deles: 300 arvores, mtry=3, min_node_size=50, honesty=0.5, sample_fraction=0.5).

In [ ]:
main_cpp = f'''#include <iostream>
#include <string>
#include <unistd.h>

#include "tree/Tree.h"
#include "prediction/DefaultPredictionStrategy.h"
#include "commons/utility.h"
#include "forest/ForestPredictor.h"
#include "forest/ForestTrainer.h"
#include "utilities/FileTestUtilities.h"
#include "utilities/ForestTestUtilities.h"
#include "forest/ForestTrainers.h"
#include "forest/ForestPredictors.h"
using namespace grf;

void update_predictions_file(const std::string& file_name,
                             const std::vector<Prediction>& predictions) {{
  std::vector<std::vector<double>> values;
  values.reserve(predictions.size());
  for (const auto& prediction : predictions) {{
    values.push_back(prediction.get_predictions());
  }}
  FileTestUtilities::write_csv_file(file_name, values);
  std::cout << "success! predictions dump to " << file_name << std::endl;
}}

int main()
{{
    auto data_vec = load_data("../hillstrom_udcf_input.txt");
    Data data(data_vec);
    data.set_outcome_index({outcome_index});
    data.set_treatment_index({{{treatment_index[0]}, {treatment_index[1]}}});

    size_t num_treatments = 2;

    ForestTrainer trainer = udcf_trainer(num_treatments, 1, true);
    ForestOptions options = ForestTestUtilities::default_options(true, 1);
    Forest forest = trainer.train(data, options);
    ForestPredictor predictor = udcf_predictor(1, num_treatments, 1);

    std::vector<Prediction> predictions = predictor.predict_oob(forest, data, false);
    update_predictions_file("../hillstrom_udcf_predictions.txt", predictions);

    return 0;
}}
'''

with open(f"{BASE}/UDCF_RCT/core/main.cpp", "w") as f:
    f.write(main_cpp)

print("main.cpp gerado com sucesso.")

## 5. Compilar (cmake + make, igual ao README original do repositorio)

In [ ]:
%%bash
cd WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/LBCF/UDCF_RCT/core
rm -rf build
mkdir build
cd build
cmake .. -DCMAKE_BUILD_TYPE=Release
make -j4

## 6. Rodar o UDCF treinado (o binario original dos autores)

In [ ]:
%%bash
cd WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/LBCF/UDCF_RCT/core/build
./UDCF_RCT

## 7. Ler o CATE estimado e resumir os resultados

In [ ]:
preds = pd.read_csv(
    f"{BASE}/UDCF_RCT/core/hillstrom_udcf_predictions.txt",
    header=None, sep=r",\s*", engine="python",
)
preds.columns = ["cate_mens_email", "cate_womens_email"]

out = pd.concat([df.reset_index(drop=True), preds], axis=1)
out.to_csv("hillstrom_udcf_cate_real.csv", index=False)

print("=== Resumo do CATE estimado (UDCF, codigo original dos autores) ===")
for nome, col in [("Mens E-Mail", "cate_mens_email"), ("Womens E-Mail", "cate_womens_email")]:
    c = out[col]
    print(f"\n{nome}")
    print(f"  media (CATE medio / ATE aproximado): {c.mean():.4f}")
    print(f"  desvio padrao entre usuarios:         {c.std():.4f}")
    print(f"  minimo / maximo:                      {c.min():.4f} / {c.max():.4f}")

naive_t1 = Y[T1 == 1].mean() - Y[(T1 == 0) & (T2 == 0)].mean()
naive_t2 = Y[T2 == 1].mean() - Y[(T1 == 0) & (T2 == 0)].mean()
print("\n=== Diferenca simples de medias (ATE naive, para conferencia) ===")
print(f"  Mens E-Mail vs controle:   {naive_t1:.4f}")
print(f"  Womens E-Mail vs controle: {naive_t2:.4f}")

print("\nArquivo salvo: hillstrom_udcf_cate_real.csv (inclui CATE por usuario)")